# Koala-Kangaroo YOLO

## Logging And Setup

We setup our logging using `wandb`
We also initialize our global variables

In [1]:
from ultralytics import settings
import os
os.environ["WANDB_DISABLE_SSL"] = "true"
settings.update({"wandb": True,
                 "clearml": False,
                 "comet": False})
YOLO_MODEL = "yolo11s.pt"
PROJECT_NAME = "koala-kangaroo"
EXPERIMENT_NAME = "exp1"
DATASET_PATH = "data/yolo_koala_kangaroo.v1-original.yolov11/data.yaml"
BEST_MODEL_PATH = f"{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt"

## Training

We specify the training path for our dataset.

For batch size, given we are training in our laptop and faced GPU out of memory problem, we leave it to Yolo to auto batch using -1 which seemed to solve the memory problem.


In [2]:
!ls -la data/yolo_koala_kangaroo.v1-original.yolov11/train/images | wc -l

242


In [3]:
from ultralytics import YOLO
from ultralytics import settings

model = YOLO(YOLO_MODEL)  # Load a pre-trained YOLO model
result = model.train(data=DATASET_PATH,
                     epochs=20, # number of training epochs
                     save_period=1, # save every epoch
                     batch=-1, # auto batch size, previously set to 16,64 and caused us to crash for GPU memory reasons
                     device=0, # use GPU 0
                     project=PROJECT_NAME, # set project name  for logging in wandb
                     name=EXPERIMENT_NAME, # set experiment name for logging in wandb
                     plots=True)

Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 2050, 3769MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/yolo_koala_kangaroo.v1-original.yolov11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

wandb: Currently logged in as: raymond-samalo (samalo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  3                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  4                  -1  1    103360  ultralytics.nn.modules.block.C3k2            [128, 256, 1, False, 0.25]    
  5                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  6                  -1  1    346112  ultralytics.nn.modules.block.C3k2            [256, 256, 1, True]           
  7                  -1  1   1180672  ultralytics

lr/pg0,▃▆██▇▇▆▆▆▅▅▄▄▃▃▃▂▂▁▁
lr/pg1,▃▆██▇▇▆▆▆▅▅▄▄▃▃▃▂▂▁▁
lr/pg2,▃▆██▇▇▆▆▆▅▅▄▄▃▃▃▂▂▁▁
metrics/mAP50(B),▃▁▂▁▂▂▂▄▅▃▅▄▅▆▇▇▇███
metrics/mAP50-95(B),▂▁▂▁▂▂▂▃▄▃▄▃▄▅▆▇▇███
metrics/precision(B),▃▁▂▁▂▂▃▅▇▃▅▄▄▅▇▇▆▇▇█
metrics/recall(B),▂▄▂▂▁▂▂▄▄▃▅▅▅▆▇▇▇███
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


## Validation

Here we are looking for mAP50 and mAP50-95 score

In [5]:
model = YOLO(BEST_MODEL_PATH)
metrics = model.val(data=DATASET_PATH, device="0")

# Retrieve precision and recall
precision = metrics.box.mp  # Mean Precision
recall = metrics.box.mr     # Mean Recall

# Calculate the F1 score
print(metrics)
if (precision + recall) > 0:
    f1_score = 2 * (precision * recall) / (precision + recall)
    print(f"F1 Score: {f1_score}")
else:
    print("Precision and recall are zero, cannot calculate F1 score.")
print(f"Precision: {precision}, Recall: {recall}")
print("Validation complete.")

Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 2050, 3769MiB)
YOLO11s summary (fused): 100 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2087.9±1113.9 MB/s, size: 43.9 KB)
val: Scanning /home/ray/Projects/25S2-C-NYP-ITI121---Applied-Deep-Learning-Assignment2/data/yolo_koala_kangaroo.v1-original.yolov11/valid/labels.cache... 66 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 66/66 133.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 3.3it/s 1.5s0.5s
                   all         66         95      0.831      0.672      0.792      0.385
              Kangaroo         31         54      0.752      0.611       0.73      0.374
                 Koala         36         41      0.909      0.732      0.854      0.396
Speed: 2.4ms preprocess, 16.2ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /home/r

## Export

In [6]:
model = YOLO(BEST_MODEL_PATH)
exported_path = model.export(format="openvino", int8=True)

Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.1+cu128 CPU (12th Gen Intel Core i5-12450HX)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s summary (fused): 100 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs

PyTorch: starting from 'koala-kangaroo/exp1/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (18.3 MB)
requirements: Ultralytics requirement ['openvino>=2024.0.0'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/50.3 MB ? eta -:--:--
   ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/50.3 MB 11.6 MB/s eta 0:00:05
   ━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/50.3 MB 11.7 MB/s eta 0:00:04
   ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/50.3 MB 11.6 MB/s eta 0:00:04
   ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━

Output()

Output()

OpenVINO: export success ✅ 28.9s, saved as 'koala-kangaroo/exp1/weights/best_int8_openvino_model/' (9.8 MB)

Export complete (29.4s)
Results saved to /home/ray/Projects/25S2-C-NYP-ITI121---Applied-Deep-Learning-Assignment2/koala-kangaroo/exp1/weights
Predict:         yolo predict task=detect model=koala-kangaroo/exp1/weights/best_int8_openvino_model imgsz=640 int8 
Validate:        yolo val task=detect model=koala-kangaroo/exp1/weights/best_int8_openvino_model imgsz=640 data=data/yolo_koala_kangaroo.v1-original.yolov11/data.yaml int8 
Visualize:       https://netron.app


## Inference

In [10]:
import ultralytics
from ultralytics import YOLO
from PIL import Image

source = 'test_image.jpg'
model = YOLO(BEST_MODEL_PATH, task='detect')
result = model(source, conf=0.5, iou=0.6)

# Visualize the results
for i, r in enumerate(result):
    print(r)
    # Plot results image
    im_bgr = r.plot()  # BGR-order numpy array
    im_rgb = Image.fromarray(im_bgr[..., ::-1])  # RGB-order PIL image

    # Show results to screen (in supported environments)
    r.show()

    # Save results to disk
    r.save(filename=f"results-{EXPERIMENT_NAME}-{i}.jpg")


image 1/1 /home/ray/Projects/25S2-C-NYP-ITI121---Applied-Deep-Learning-Assignment2/test_image.jpg: 640x640 2 Kangaroos, 1 Koala, 14.6ms
Speed: 1.7ms preprocess, 14.6ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)
ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'Kangaroo', 1: 'Koala'}
obb: None
orig_img: array([[[ 88, 231, 192],
        [ 83, 215, 178],
        [ 94, 205, 173],
        ...,
        [ 95, 130, 116],
        [ 96, 131, 117],
        [ 97, 132, 118]],

       [[ 77, 216, 178],
        [ 73, 204, 167],
        [ 87, 198, 166],
        ...,
        [ 92, 127, 113],
        [ 93, 128, 114],
        [ 94, 129, 115]],

       [[ 68, 200, 163],
        [ 69, 193, 157],
        [ 82, 193, 161],
        ...,
        [ 89, 124, 110],
        [ 89, 124, 110],
        [ 89, 124, 110]],

       ...,

       [[ 56,  66,  66],
        [ 60,  71,  69],
        [ 61,  69,


(gthumb:23632): Gtk-WARNING **: 11:56:26.654: Getting screensaver status failed: GDBus.Error:org.freedesktop.DBus.Error.ServiceUnknown: The name org.gnome.Shell.ScreenShield was not provided by any .service files


## Video